# `single_window()` — `jimenez-munoz-2014-sc`

Estimar LST por Jiménez Single-channel com entradas preparadas.

**Cadeia deste exemplo:** BT10 + radiância B10 + ε10 + PWV + QA → Jiménez SC → LST.

Este notebook executa uma etapa sobre arquivos já preparados. **Se você tem apenas os arquivos originais Level-1, prepare as entradas indicadas na tabela antes de executar este método.**

Execute apenas os **quatro blocos de código**, de cima para baixo. Os blocos ao final, em “Consulta”, são alternativas para leitura, e não uma continuação obrigatória.


## Antes de executar: separe estes arquivos

| Campo a preencher no passo 1 | Arquivo que você deve escolher | Unidade / tipo | Como obter se ainda falta |
| --- | --- | --- | --- |
| `emissividade_path` | Emissividade B10; por exemplo, `emissividade_B10.tif` | Adimensional, (0,1] | [Rongali](emissivity__rongali-2018.ipynb) ou [Avdan](emissivity__avdan-2016.ipynb), conforme o modelo escolhido |
| `bt10_path` | Temperatura de brilho B10; por exemplo, `BT_B10_K.tif` | K | [Temperatura de brilho](brightness.ipynb) com `banda=10` |
| `radiancia10_path` | Radiância térmica B10; por exemplo, `radiancia_B10.tif` | Radiância TOA; W m⁻² sr⁻¹ µm⁻¹ | [Radiância TOA](toa_radiance.ipynb) com `banda=10`, mesma calibração da BT10 |
| `pwv_path` | Vapor d’água integrado na coluna; por exemplo, `PWV.tif` | g/cm² | [Wang](water_vapor__wang-2015.ipynb) ou [Ren](water_vapor__ren-2015.ipynb) |
| `qa_pixel_path` | Qualidade dos pixels; por exemplo, `PRODUTO_QA_PIXEL.TIF` | Bits inteiros, sem calibração | Arquivo original QA_PIXEL da mesma cena e na mesma grade |
| `qa_radsat_path` | Saturação/qualidade geométrica; por exemplo, `PRODUTO_QA_RADSAT.TIF` | Bits inteiros, sem calibração | Arquivo original QA_RADSAT da mesma cena e na mesma grade |

Todos os GeoTIFFs devem representar a mesma cena e ter dimensões, CRS e transformação idênticos. Para recortes, use a mesma janela nas QA. As unidades são verificadas pelo usuário: a leitura do arquivo não comprova sua grandeza física.

**Configuração desta execução:** BT10 e radiância B10 são arquivos distintos e ambos necessários. Este exemplo usa PWV raster; no modo padrão, o suporte operacional é 0 a 3 g/cm². Escolha a unidade apenas para a saída.

**Ambiente:** instale `python -m pip install -e ".[examples]"` na pasta da biblioteca e selecione o kernel desse ambiente no VS Code/Jupyter.


## Passo 1 — Preencher caminhos e escolher a saída

Substitua cada `CAMINHO_...` pelo caminho completo do seu arquivo, mantendo `Path(...)`. No Windows, escreva, por exemplo, `Path(r"C:\dados\radiancia_B4.tif")`. Preencha todos os campos da tabela, inclusive as duas QA.

BT10 e radiância B10 são arquivos distintos e ambos necessários. Este exemplo usa PWV raster; no modo padrão, o suporte operacional é 0 a 3 g/cm². Escolha a unidade apenas para a saída.

Ajuste `saida` para a pasta desejada. Caminhos relativos partem da pasta de trabalho do kernel; a célula imprime o caminho absoluto. Em nova execução com o mesmo destino, o GeoTIFF é substituído.


In [ ]:
from pathlib import Path
import numpy as np
import rasterio

# EDITE OS CAMINHOS E AS CONFIGURAÇÕES ABAIXO; mantenha o método deste notebook.
from pylstemp import read_quality_mask
from pylstemp import single_window

emissividade_path = Path("CAMINHO_DA_EMISSIVIDADE_B10.tif")
bt10_path = Path("CAMINHO_DA_BT10_K.tif")
radiancia10_path = Path("CAMINHO_DA_RADIANCIA_B10.tif")
pwv_path = Path("CAMINHO_DO_PWV_G_CM2.tif")
qa_pixel_path = Path("CAMINHO_DO_QA_PIXEL.tif")
qa_radsat_path = Path("CAMINHO_DO_QA_RADSAT.tif")

saida = Path("resultados/jimenez_munoz_2014_sc")
unidade = "kelvin"  # ou "celsius"; a BT10 de entrada permanece em K

# Confira os arquivos antes de passar ao passo 2.
entradas_obrigatorias = {
    "emissividade_path": emissividade_path,
    "bt10_path": bt10_path,
    "radiancia10_path": radiancia10_path,
    "pwv_path": pwv_path,
    "qa_pixel_path": qa_pixel_path,
    "qa_radsat_path": qa_radsat_path,
}
pendentes = [f"{nome}: {caminho}" for nome, caminho in entradas_obrigatorias.items()
             if not caminho.is_file()]
if pendentes:
    raise FileNotFoundError("Preencha os caminhos de entrada com arquivos existentes:\n"
                            + "\n".join(pendentes))
print("Arquivos encontrados. Próximo passo: ler os dados e aplicar QA.")
print("Pasta de saída:", saida.resolve())


## Passo 2 — Ler as entradas, conferir a grade e aplicar QA

Execute esta célula depois de confirmar os caminhos. Ela carrega os dados, confere seu alinhamento, lê QA_PIXEL/QA_RADSAT com `read_quality_mask()` e combina QA com nodata em `mascara`. `True` indica pixel excluído. A leitura não recalibra dados prontos.

Água é preservada na política QA deste exemplo (`exclude_water=False`). A célula imprime as dimensões e as contagens de exclusão.


In [ ]:
arquivos = {
    "e10": emissividade_path,
    "bt10": bt10_path,
    "rad10": radiancia10_path,
    "pwv": pwv_path,
}
dados = {}
grade = None
for nome, caminho in arquivos.items():
    with rasterio.open(caminho) as src:
        grade_atual = (src.shape, src.crs, src.transform)
        if grade is None:
            grade = grade_atual
            perfil = src.profile.copy()
        elif grade_atual != grade:
            raise ValueError(f"A imagem {nome} não está na mesma grade das outras entradas.")
        dados[nome] = src.read(1, masked=True).astype(float).filled(np.nan)

e10, bt10, rad10, pwv = (dados[nome] for nome in ("e10", "bt10", "rad10", "pwv"))
mascara = ~np.isfinite(e10) | ~np.isfinite(bt10) | ~np.isfinite(rad10) | ~np.isfinite(pwv)
# Ler QA na grade da primeira imagem e combinar com nodata antes do método.
mascara_qa = read_quality_mask(
    qa_pixel_path, qa_radsat_path,
    reference_path=next(iter(arquivos.values())),
    exclude_water=False,
)
mascara |= mascara_qa

print("Dimensões:", mascara.shape)
print("Pixels excluídos por QA:", int(mascara_qa.sum()))
print("Pixels excluídos por QA ou nodata:", int(mascara.sum()), "de", mascara.size)


## Passo 3 — Calcular a temperatura

Execute a chamada abaixo. As variáveis são matrizes carregadas no passo 2; os caminhos de arquivo foram usados no passo 1. `mask=mascara` aplica a seleção QA/nodata ao cálculo.


In [ ]:
lst = single_window(
    lst_method="jimenez-munoz-2014-sc",
    emissivity_10=e10,
    brightness_band_10=bt10,
    radiance_band_10=rad10,
    water_vapor=pwv,
    mask=mascara,
    unit=unidade,
)

## Passo 4 — Salvar e usar a saída

Execute esta célula para gravar o GeoTIFF em `saida`, preservando a grade. Confira o caminho absoluto e a quantidade de pixels válidos impressos. Pixels inválidos são gravados como NaN.

**Próximo uso:** Comparar a LST com a referência de campo. A preparação de PWV Wang/Ren usa também BT11, embora a equação final SC use B10.


In [ ]:
saida.mkdir(parents=True, exist_ok=True)
perfil.update(driver="GTiff", dtype="float32", count=1, nodata=np.nan, compress="deflate")
with rasterio.open(saida / "jimenez_munoz_2014_sc.tif", "w", **perfil) as dst:
    dst.write(np.asarray(lst, dtype="float32"), 1)
    dst.update_tags(units=unidade, lst_method="jimenez-munoz-2014-sc")
print("Pixels válidos:", int(np.isfinite(lst).sum()), "de", lst.size)
print("Saída:", saida.resolve())

## Como conferir a execução

Abra o GeoTIFF salvo em um SIG e confira a posição, a unidade e a banda informadas acima. Compare a contagem de pixels válidos com a área disponível: QA pode excluir pixels, e os critérios do próprio método podem produzir lacunas adicionais.

| Se acontecer | O que conferir |
| --- | --- |
| `FileNotFoundError` no passo 1 | Preencha os caminhos listados no erro, inclusive MTL quando solicitado e ambas as QA. |
| Erro de grade no passo 2 | Prepare arquivos com a mesma projeção, transformação, dimensões e recorte. Não redimensione as QA por interpolação contínua. |
| Erro de domínio no passo 3 | Confira a grandeza/unidade e os requisitos do método. |
| Saída com NaN | Confira as contagens QA/nodata e o suporte do método; NaN não deve ser substituído por temperatura inventada. |

[Voltar ao índice e à ordem de execução](README.md).


## Consulta — opções e detalhes do método

O roteiro executável termina acima. Leia os tópicos abaixo para adaptar a entrada ou consultar parâmetros; não execute todas as alternativas em sequência sobre a mesma saída.


### Entradas da função e opções

### Para usar emissividade pronta — caminho recomendado

Forneça **estas quatro entradas**:

| Entrada da função | O que fornecer | Unidade / domínio |
|---|---|---|
| `emissivity_10` | Imagem de emissividade específica da banda 10, já calculada | Adimensional, **(0,1]** |
| `brightness_band_10` | Imagem de temperatura de brilho TOA da banda 10 | **K** |
| `radiance_band_10` | Imagem de radiância TOA da banda 10, correspondente à mesma calibração da BT10 | **W m⁻² sr⁻¹ µm⁻¹** |
| `water_vapor` | Vapor d’água integrado na coluna (PWV), como imagem alinhada ou valor único representativo da área | **g/cm²**, de **0 a 3**, inclusive |

**Todas as imagens devem estar na mesma grade:** projeção, resolução, posição dos pixels e dimensões. BT10 e radiância B10 são duas grandezas diferentes; o método exige ambas.

**Já possui essas entradas?** Pode usar a chamada abaixo. B4, B5, B11 e MTL não entram nessa chamada com dados prontos. O método que produziu a emissividade foi escolhido antes; não há outro método de emissividade a selecionar aqui.

### Se ainda não tem emissividade pronta

Como alternativa, fornecer **vermelho + NIR calibrados** e escolher `emissivity_method`, omitindo `emissivity_10`. As outras três entradas continuam necessárias. A [consulta de alternativas](#alternativas-de-uso) mostra essa chamada; não combine as duas formas de emissividade.

### O que é opcional

- **`mask`:** máscara booleana de exclusão, alinhada às imagens. `True` exclui o pixel; `False` permite avaliar. Neste exemplo é preenchida com QA/nodata no passo 2; não fornecer QA_PIXEL bruta.
- **`unit`:** saída em `"kelvin"` (padrão) ou `"celsius"`. BT10 entra sempre em K.
- **`return_diagnostics`:** `False` (padrão) retorna LST; `True` retorna LST e intermediários.

### Alternativas de uso

### 5.1. Calcular a emissividade dentro de `single_window()`

Escolha esta chamada **em lugar** da chamada com `emissivity_10`. Forneça as duas bandas ópticas na mesma grandeza e grade e selecione `emissivity_method`. As opções internas são `"avdan-2016"` e `"rongali-2018"`; se omitido ou `None`, o padrão é Avdan. O NDVI é calculado internamente.

```python
lst = single_window(
    lst_method="jimenez-munoz-2014-sc",
    emissivity_method="avdan-2016",
    band_4_red=red,
    band_5_nir=nir,
    optical_quantity="radiance",
    brightness_band_10=bt10,
    radiance_band_10=rad10,
    water_vapor=pwv,
    mask=mascara,
    unit="kelvin",
)
```

**Nesta alternativa não fornecer `emissivity_10`.** Sobrino 2004 TM6 não é um modelo de B10.

### 5.2. Usar um valor único de PWV

`water_vapor` aceita um número representativo da área, além do raster. Na chamada de LST, você pode substituir `water_vapor=pwv` por `water_vapor=pwv_da_area`, com esse valor em **g/cm²**, finito e em **[0,3]**. Ele será aplicado à imagem inteira; não é equivalente a usar PWV variável por pixel.

A função não escolhe esse valor nem calcula média automaticamente. Se optar pela média de um raster, registrar a área e os pixels aceitos usados para essa média. Não há PWV padrão para este método.

### 5.3. Mudar a saída ou pedir diagnósticos

Na chamada de LST, `unit="celsius"` muda somente a unidade da saída. `return_diagnostics=True` retorna um dicionário; use `resultado["lst"]` para acessar a imagem de LST e salvá-la.

Os diagnósticos são `lst`, `ndvi`, `emissivity_10`, `invalid_mask`, `units` e `optical_contract`. Com emissividade pronta, `ndvi` é `None`; a função não o recalcula. `optical_contract` registra declarações de entrada e não certifica a validade do modelo na cena.

### Referência completa dos parâmetros

### Rota com emissividade pronta

A chamada completa abaixo reúne os parâmetros aplicáveis a esta rota. A tabela segue a mesma ordem.

```python
resultado = single_window(
    lst_method="jimenez-munoz-2014-sc",
    emissivity_10=e10,
    brightness_band_10=bt10,
    radiance_band_10=rad10,
    water_vapor=pwv,
    mask=mascara,
    unit="kelvin",
    return_diagnostics=False,
)
```

| Parâmetro | Condição neste método | Padrão da API | Unidade | Detalhamento |
|---|---|---|---|---|
| `lst_method` | Selecionar explicitamente | `"mono-window-2016"` | Seletor | Usar `"jimenez-munoz-2014-sc"`; omitir o seletor escolheria outro algoritmo. |
| `emissivity_10` | Obrigatório nesta rota | `None` | Adimensional, (0,1] | Array da banda 10, compatível com o sensor e alinhado à BT10. Omitir `emissivity_method` e as bandas ópticas; não é recalculado. |
| `brightness_band_10` | Obrigatório nas duas rotas | Sem padrão | K | Temperatura de brilho TOA B10. Não usar DN, radiância, LST ou Celsius. |
| `radiance_band_10` | Obrigatório em Jiménez SC | `None` | W m⁻² sr⁻¹ µm⁻¹ | Radiância TOA B10 da mesma calibração e dos mesmos pixels da BT10; não pode ser substituída por BT10. |
| `water_vapor` | Obrigatório em Jiménez SC | `None` | g/cm², [0,3] | Escalar ou array alinhado. Escalar fora do domínio gera erro; pixels raster fora do domínio ou não finitos produzem NaN. Não é estimado internamente. |
| `mask` | Opcional na API; fornecido neste exemplo | `None` | Booleano | Array alinhado: True exclui e False permite avaliar. Não cria máscara de nuvens nem interpreta QA automaticamente. A máscara final também incorpora rejeições das entradas. |
| `unit` | Opcional | `"kelvin"` | K ou °C | `"kelvin"` ou `"celsius"`; altera somente a saída. BT10 entra em K nas duas opções. |
| `return_diagnostics` | Opcional | `False` | Booleano | False retorna array; True retorna dicionário. `ndvi=None` com emissividade pronta. |

### Parâmetros exclusivos da alternativa por bandas

Eles substituem `emissivity_10`. As outras entradas e opções permanecem as da tabela acima. A ordem abaixo acompanha a chamada da consulta de alternativas.1.

| Parâmetro | Condição nesta rota | Padrão da API | Unidade | Detalhamento |
|---|---|---|---|---|
| `emissivity_method` | Opcional; escolher para mudar o modelo | `None` → `"avdan-2016"` | Seletor | `"avdan-2016"` ou `"rongali-2018"`. Omitir quando fornecer emissividade pronta. |
| `band_4_red` | Obrigatório junto com NIR | `None` | Grandeza óptica calibrada; radiância TOA em W m⁻² sr⁻¹ µm⁻¹ no exemplo | Banda vermelha, na mesma grade e convenção radiométrica do NIR, compatível com o modelo de emissividade. |
| `band_5_nir` | Obrigatório junto com vermelho | `None` | Mesma grandeza e unidade do vermelho | Banda NIR para o cálculo interno de NDVI. Não misturar DN com radiância ou reflectância. |
| `optical_quantity` | Opcional; recomendado declarar nesta rota | `None` | Seletor, sem unidade | `"radiance"`, `"reflectance"`/`"toa_reflectance"` ou `"surface_reflectance"`. Apenas declara a grandeza; não converte dados nem faz correção atmosférica. Não transforma emissividade pronta em radiância. |

### Aplicabilidade e limites

- Os coeficientes de recuperação são de Landsat 8. A aplicação a Landsat 9 é transferência experimental, separada da calibração radiométrica pelo MTL próprio de cada cena.
- A biblioteca restringe este ajuste polinomial a **0–3 g/cm² de PWV**. Lacunas não são preenchidas automaticamente. Mudanças de estimador de PWV ou de emissividade precisam manter os domínios e as unidades exigidos.
- A execução do código e a equivalência entre rotas não demonstram acurácia absoluta de LST. A compatibilidade dos modelos com a cena e a validação independente continuam necessárias.

[Jiménez-Muñoz et al. (2014)](https://doi.org/10.1109/LGRS.2014.2312032) · [Fichas metodológicas](../docs/method-cards.md) · [Índice dos exemplos](README.md).